In [1]:
import polars as pl
import datetime
from polars import DataFrame

In [2]:
dataset = '../data/HI-Medium_Trans.csv'

trans = (
    pl.read_csv(dataset)
    .with_columns(
        pl.col("Timestamp").str.strptime(pl.Datetime, '%Y/%m/%d %H:%M', strict=True)
    )
    .sort('Timestamp')
)

In [3]:
trans.head()

Timestamp,From Bank,From,To Bank,To,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
datetime[μs],i64,str,i64,str,f64,str,f64,str,str,i64
2022-09-01 00:00:00,1046,"""800A37D90""",274159,"""820C04F20""",26.42,"""US Dollar""",26.42,"""US Dollar""","""Credit Card""",0
2022-09-01 00:00:00,21418,"""800AB4DE0""",21418,"""800AB4DE0""",23.61,"""US Dollar""",23.61,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:00:00,32248,"""80056BBD0""",11,"""800BAFE20""",12373.74,"""US Dollar""",12373.74,"""US Dollar""","""ACH""",0
2022-09-01 00:00:00,11798,"""80145BAF0""",11798,"""80145BAF0""",4981.27,"""US Dollar""",4981.27,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:00:00,1924,"""800CCE420""",1924,"""800CCE420""",22.94,"""US Dollar""",22.94,"""US Dollar""","""Reinvestment""",0


In [4]:
zero = trans['Timestamp'].min()
hundred = trans['Timestamp'].max()

In [5]:
diff = hundred - zero
days = diff.days
sixty = zero + datetime.timedelta(days=days * 0.6)
eighty = zero + datetime.timedelta(days=days * 0.8)
hundred = zero + datetime.timedelta(days=days)

print(zero, sixty, eighty, hundred)

2022-09-01 00:00:00 2022-09-17 04:48:00 2022-09-22 14:24:00 2022-09-28 00:00:00


# Transform accounts to IDs

In [6]:
nodes = (
    trans
    .select(pl.col('From').alias('Acc'))
    .vstack(trans.select(pl.col('To').alias('Acc')))
    .unique()
    .with_row_index('Node ID')
)

ssl = (
    trans.join(nodes, left_on='From', right_on='Acc')
    .join(nodes, left_on='To', right_on='Acc', suffix='_to')
    .drop('From', 'To')
    .rename({
        'Node ID': 'From',
        'Node ID_to': 'To',
    })
)

In [11]:
ssl.head()

Timestamp,From Bank,To Bank,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,From,To
datetime[μs],i64,i64,f64,str,f64,str,str,i64,u32,u32
2022-09-01 00:00:00,1046,274159,26.42,"""US Dollar""",26.42,"""US Dollar""","""Credit Card""",0,950276,1143989
2022-09-01 00:00:00,21418,21418,23.61,"""US Dollar""",23.61,"""US Dollar""","""Reinvestment""",0,1311770,1311770
2022-09-01 00:00:00,32248,11,12373.74,"""US Dollar""",12373.74,"""US Dollar""","""ACH""",0,1820335,1575420
2022-09-01 00:00:00,11798,11798,4981.27,"""US Dollar""",4981.27,"""US Dollar""","""Reinvestment""",0,1214029,1214029
2022-09-01 00:00:00,1924,1924,22.94,"""US Dollar""",22.94,"""US Dollar""","""Reinvestment""",0,1894020,1894020


In [38]:
print(f"Number of unique nodes: {nodes.count()}")

Number of unique nodes: shape: (1, 2)
┌─────────┬─────────┐
│ Node ID ┆ Acc     │
│ ---     ┆ ---     │
│ u32     ┆ u32     │
╞═════════╪═════════╡
│ 2076999 ┆ 2076999 │
└─────────┴─────────┘


# Global vars

In [12]:
every = '2d'

In [13]:
def _prep(df: pl.DataFrame) -> pl.LazyFrame:
    return (
        df.lazy()
        .with_columns(
            pl.col('Timestamp').dt.truncate(every).alias("window_start")
        )
        .filter(
             pl.col('Timestamp').is_not_null()
             & pl.col('From').is_not_null()
             & pl.col('To').is_not_null()
        )
    )

In [14]:
_prep(ssl).collect()

Timestamp,From Bank,To Bank,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,From,To,window_start
datetime[μs],i64,i64,f64,str,f64,str,str,i64,u32,u32,datetime[μs]
2022-09-01 00:00:00,1046,274159,26.42,"""US Dollar""",26.42,"""US Dollar""","""Credit Card""",0,950276,1143989,2022-09-01 00:00:00
2022-09-01 00:00:00,21418,21418,23.61,"""US Dollar""",23.61,"""US Dollar""","""Reinvestment""",0,1311770,1311770,2022-09-01 00:00:00
2022-09-01 00:00:00,32248,11,12373.74,"""US Dollar""",12373.74,"""US Dollar""","""ACH""",0,1820335,1575420,2022-09-01 00:00:00
2022-09-01 00:00:00,11798,11798,4981.27,"""US Dollar""",4981.27,"""US Dollar""","""Reinvestment""",0,1214029,1214029,2022-09-01 00:00:00
2022-09-01 00:00:00,1924,1924,22.94,"""US Dollar""",22.94,"""US Dollar""","""Reinvestment""",0,1894020,1894020,2022-09-01 00:00:00
…,…,…,…,…,…,…,…,…,…,…,…
2022-09-28 11:59:00,211767,219853,7339.28,"""Yuan""",7339.28,"""Yuan""","""ACH""",1,1538965,2054979,2022-09-27 00:00:00
2022-09-28 12:14:00,211767,211767,5869.16,"""Euro""",6877.38,"""US Dollar""","""ACH""",0,1538965,1538965,2022-09-27 00:00:00
2022-09-28 12:14:00,211767,130596,5869.16,"""Euro""",5869.16,"""Euro""","""ACH""",1,1538965,209851,2022-09-27 00:00:00


# Temporal stats

In [39]:
def add_temporal_stats(df: pl.DataFrame) -> pl.DataFrame:
    df = _prep(df).select(
        pl.col('window_start'),
        pl.col('Timestamp'),
        pl.col('From'),
        pl.col('To'),
    )

    in_gaps = (
        df.sort(["window_start", "To", "Timestamp"])
        .group_by(["window_start", "To"])
        .agg(
            pl.col('Timestamp').diff().dt.total_seconds().alias('gap')
        )
        .with_columns(
            pl.col("gap").list.drop_nulls().alias("gap")
        )
        .with_columns([
            pl.col("gap").list.mean().alias("mu_gap_in_sec"),
            pl.col("gap").list.var().alias("var_gap_in_sec"),
        ])
        .select(["window_start", 'To', "mu_gap_in_sec", "var_gap_in_sec"])
        .rename({'To': 'Node'})
    )

    out_gaps = (
        df.sort(["window_start", "From", "Timestamp"])
        .group_by(["window_start", "From"])
        .agg(
            pl.col('Timestamp').diff().dt.total_seconds().alias('gap')
        )
        .with_columns(
            pl.col("gap").list.drop_nulls().alias("gap")
        )
        .with_columns([
            pl.col("gap").list.mean().alias("mu_gap_out_sec"),
            pl.col("gap").list.var().alias("var_gap_out_sec"),
        ])
        .select(["window_start", 'From', "mu_gap_out_sec", "var_gap_out_sec"])
        .rename({'From': 'Node'})
    )

    return (
        in_gaps.join(out_gaps, on=["Node", 'window_start'], how="full", coalesce=True)
        .collect()
    )

In [40]:
temporal_features = add_temporal_stats(ssl)

In [41]:
temporal_features

window_start,Node,mu_gap_in_sec,var_gap_in_sec,mu_gap_out_sec,var_gap_out_sec
datetime[μs],u32,f64,f64,f64,f64
2022-09-15 00:00:00,1486638,null,null,null,null
2022-09-13 00:00:00,668634,22440.0,1.4725e9,null,null
2022-09-01 00:00:00,290756,24342.857143,1.9297e9,54300.0,null
2022-09-11 00:00:00,1079620,25752.0,3.1147e9,null,null
2022-09-11 00:00:00,821487,180.0,null,6333.333333,3.2523e8
…,…,…,…,…,…
2022-09-15 00:00:00,1530147,null,null,21248.571429,3.3527e8
2022-09-13 00:00:00,1073502,null,null,null,null
2022-09-05 00:00:00,1408649,null,null,29260.0,2.3069e9


# Fraudulent patterns

## 2-cycles

In [46]:
def count_reciprocal_neighbours(df: DataFrame) -> DataFrame:
    lf = _prep(df)

    edges = lf.select(["window_start", "From", "To"]).unique().filter(pl.col("From") != pl.col("To"))

    recip = (
        edges.join(
            edges,
            how="inner",
            left_on=["window_start", "From", "To"],
            right_on=["window_start", "To", "From"],
        )
        .select(["window_start", "From", "To"])
        .unique()
        .group_by(["window_start", "From"])
        .agg(
            pl.col("To").n_unique().alias("r_2cycle")
        )
        .rename({"From": "Node"})
    )

    nodes = (
        pl.concat([
            lf.select(["window_start", pl.col("From").alias("Node")]),
            lf.select(["window_start", pl.col("To").alias("Node")]),
        ])
        .unique()
    )

    return (
        nodes.join(recip, on=["window_start", "Node"], how="left")
        .with_columns(
            pl.col("r_2cycle").fill_null(0).cast(pl.Int64)
        )
        .collect()
    )


In [47]:
two_cycles = count_reciprocal_neighbours(ssl)

In [48]:
two_cycles

window_start,Node,r_2cycle
datetime[μs],u32,i64
2022-09-01 00:00:00,1863431,0
2022-09-01 00:00:00,1150128,0
2022-09-07 00:00:00,516338,0
2022-09-05 00:00:00,1726037,0
2022-09-15 00:00:00,998716,0
…,…,…
2022-09-01 00:00:00,1725707,0
2022-09-15 00:00:00,381566,0
2022-09-11 00:00:00,987511,0


In [49]:
two_cycles.describe()

statistic,window_start,Node,r_2cycle
str,str,f64,f64
"""count""","""9337964""",9.337964e6,9.337964e6
"""null_count""","""0""",0.0,0.0
"""mean""","""2022-09-07 18:20:54.673845""",1.0387e6,0.001221
"""std""",null,599722.740056,0.035827
"""min""","""2022-09-01 00:00:00""",0.0,0.0
"""25%""","""2022-09-03 00:00:00""",519187.0,0.0
"""50%""","""2022-09-09 00:00:00""",1.038889e6,0.0
"""75%""","""2022-09-13 00:00:00""",1.558066e6,0.0
"""max""","""2022-09-27 00:00:00""",2.076998e6,3.0


## Ego profile

In [50]:
EPS = 1e-12

def compute_ego_profiles(trans: DataFrame) -> DataFrame:
    lf = _prep(trans)

    out_stats = (
        lf.group_by(["window_start", "From"])
        .agg(
            pl.len().alias("deg_out"),
            pl.col("To").n_unique().alias("fan_out"),
            pl.col("Amount Paid").sum().alias("vol_out"),
        )
        .rename({"From": "Node"})
    )

    in_stats = (
        lf.group_by(["window_start", "To"])
        .agg(
            pl.len().alias("deg_in"),
            pl.col("From").n_unique().alias("fan_in"),
            pl.col("Amount Received").sum().alias("vol_in"),
        )
        .rename({"To": "Node"})
    )

    ego = (
        out_stats.join(in_stats, on=["window_start", "Node"], how="full")
        .with_columns(
            pl.col("deg_out").fill_null(0).cast(pl.Int64),
            pl.col("fan_out").fill_null(0).cast(pl.Int64),
            pl.col("vol_out").fill_null(0.0),
            pl.col("deg_in").fill_null(0).cast(pl.Int64),
            pl.col("fan_in").fill_null(0).cast(pl.Int64),
            pl.col("vol_in").fill_null(0.0),
        )
        .with_columns(
            ((pl.col("vol_in") - pl.col("vol_out"))/ (pl.col("vol_in") + pl.col("vol_out") + pl.lit(EPS))).alias("flow_imbalance")
        )
        .with_columns(
            pl.when(pl.col('window_start').is_null()).then(pl.col('window_start_right')).otherwise(pl.col('window_start')).alias('window_start'),
            pl.when(pl.col('Node').is_null()).then(pl.col('Node_right')).otherwise(pl.col('Node')).alias('Node'),
        )
        .drop('Node_right', 'window_start_right')
    )

    per_cur_in = (
        lf.group_by(["window_start", "To", "Receiving Currency"])
        .agg(pl.col("Amount Received").sum().alias("vol_in_cur"))
        .rename({"To": "Node", "Receiving Currency": "currency"})
    )

    mix_in = (
        per_cur_in.join(
            per_cur_in.group_by(["window_start", "Node"])
            .agg(pl.col("vol_in_cur").sum().alias("vol_in_sum")),
            on=["window_start", "Node"],
        )
        .with_columns((pl.col("vol_in_cur") / (pl.col("vol_in_sum") + pl.lit(EPS))).alias("p"))
        .group_by(["window_start", "Node"])
        .agg([
            pl.len().alias("n_currencies_in"),
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("currency_entropy_in"),
            pl.col("p").max().alias("top_currency_share_in"),
        ])
    )

    per_cur_out = (
        lf.group_by(["window_start", "From", "Payment Currency"])
        .agg(pl.col("Amount Paid").sum().alias("vol_out_cur"))
        .rename({"From": "Node", "Payment Currency": "currency"})
    )

    mix_out = (
        per_cur_out.join(
            per_cur_out.group_by(["window_start", "Node"])
            .agg(pl.col("vol_out_cur").sum().alias("vol_out_sum")),
            on=["window_start", "Node"],
        )
        .with_columns((pl.col("vol_out_cur") / (pl.col("vol_out_sum") + pl.lit(EPS))).alias("p"))
        .group_by(["window_start", "Node"])
        .agg([
            pl.len().alias("n_currencies_out"),
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("currency_entropy_out"),
            pl.col("p").max().alias("top_currency_share_out"),
        ])
    )

    full = (
        ego.join(mix_in, on=["window_start", "Node"], how="left")
        .join(mix_out, on=["window_start", "Node"], how="left")
        .with_columns(
            pl.col("n_currencies_in").fill_null(0).cast(pl.Int64),
            pl.col("currency_entropy_in").fill_null(0.0),
            pl.col("top_currency_share_in").fill_null(0.0),

            pl.col("n_currencies_out").fill_null(0).cast(pl.Int64),
            pl.col("currency_entropy_out").fill_null(0.0),
            pl.col("top_currency_share_out").fill_null(0.0),
        )
    )

    return full.collect()

In [53]:
ego_profile = compute_ego_profiles(ssl)

In [54]:
ego_profile

window_start,Node,deg_out,fan_out,vol_out,deg_in,fan_in,vol_in,flow_imbalance,n_currencies_in,currency_entropy_in,top_currency_share_in,n_currencies_out,currency_entropy_out,top_currency_share_out
datetime[μs],u32,i64,i64,f64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64
2022-09-09 00:00:00,2004072,0,0,0.0,3,2,478.47,1.0,1,-9.9787e-13,1.0,0,0.0,0.0
2022-09-11 00:00:00,869769,0,0,0.0,4,1,435.2,1.0,1,-9.9765e-13,1.0,0,0.0,0.0
2022-09-13 00:00:00,1969402,4,1,1060.68,4,1,367.32,-0.485546,1,-9.9720e-13,1.0,1,-9.9920e-13,1.0
2022-09-11 00:00:00,984491,0,0,0.0,2,1,13630.73,1.0,1,-9.9987e-13,1.0,0,0.0,0.0
2022-09-13 00:00:00,354060,0,0,0.0,8,2,15266.3,1.0,1,-9.9987e-13,1.0,0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2022-09-15 00:00:00,538406,1,1,395.3,0,0,0.0,-1.0,0,0.0,0.0,1,-9.9742e-13,1.0
2022-09-15 00:00:00,1060131,10,2,20.129332,0,0,0.0,-1.0,0,0.0,0.0,1,-9.5035e-13,1.0
2022-09-15 00:00:00,719593,1,1,169.74,0,0,0.0,-1.0,0,0.0,0.0,1,-9.9409e-13,1.0


# Flow-aware positional prediction

In [20]:
EPS = 1e-12

def flow_targets_out_entropy_count(trans: DataFrame, k2: bool = True) -> DataFrame:
    lf = _prep(trans).select(["window_start", "From", "To"])

    # Edge weight = count of transactions (currency invariant)
    W = (
        lf.group_by(["window_start", "From", "To"])
        .agg(pl.len().alias("w"))
    )

    out_sum = (
        W.group_by(["window_start", "From"])
        .agg(pl.col("w").sum().alias("out_wsum"))
    )

    P = (
        W.join(out_sum, on=["window_start", "From"], how="left")
        .with_columns((pl.col("w") / (pl.col("out_wsum") + pl.lit(EPS))).alias("p"))
        .select(["window_start", "From", "To", "p"])
    )

    one = (
        P.group_by(["window_start", "From"])
        .agg(
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("H_out_1_cnt"),
            pl.len().alias("supp_out_1_cnt"),
            pl.col("p").max().alias("pmax_out_1_cnt"),
        )
        .rename({"From": "Node"})
    )

    if not k2:
        return one.collect()

    # 2-step distribution via mid join: sum_mid P(src->mid)*P(mid->dst)
    P1 = P.rename({"From": "src", "To": "mid", "p": "p1"})
    P2 = P.rename({"From": "mid", "To": "dst", "p": "p2"})

    two_step = (
        P1.join(P2, on=["window_start", "mid"], how="inner")
        .with_columns((pl.col("p1") * pl.col("p2")).alias("p2step"))
        .group_by(["window_start", "src", "dst"])
        .agg(pl.col("p2step").sum().alias("p"))
    )

    two = (
        two_step.group_by(["window_start", "src"])
        .agg(
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("H_out_2_cnt"),
            pl.len().alias("supp_out_2_cnt"),
            pl.col("p").max().alias("pmax_out_2_cnt"),
        )
        .rename({"src": "Node"})
    )

    return one.join(two, on=["window_start", "Node"], how="left").collect()


def currency_mix_out(trans: DataFrame) -> DataFrame:
    lf = _prep(trans).select(["window_start", "From", "Payment Currency", "Amount Paid"])

    per_cur = (
        lf.group_by(["window_start", "From", "Payment Currency"])
        .agg(pl.col("Amount Paid").sum().alias("vol_out_cur"))
        .rename({"From": "Node", "Payment Currency": "currency"})
    )

    mix = (
        per_cur.join(
            per_cur.group_by(["window_start", "Node"])
            .agg(pl.col("vol_out_cur").sum().alias("vol_out_sum")),
            on=["window_start", "Node"],
        )
        .with_columns((pl.col("vol_out_cur") / (pl.col("vol_out_sum") + pl.lit(EPS))).alias("p"))
        .group_by(["window_start", "Node"])
        .agg([
            pl.count().alias("n_currencies_out"),
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("currency_entropy_out"),
            pl.col("p").max().alias("top_currency_share_out"),
        ])
    )

    return mix.collect()


def flow_heads_A_B(trans: DataFrame, k2: bool = True) -> DataFrame:
    head_a = flow_targets_out_entropy_count(trans, k2=k2)   # (window_start, node, ...)
    head_b = currency_mix_out(trans)                        # (window_start, node, ...)

    # Node universe: all active nodes in the window (senders or receivers)
    lf = _prep(trans).select(["window_start", "From", "To"])
    nodes = (
        pl.concat([
            lf.select(["window_start", pl.col("From").alias("Node")]),
            lf.select(["window_start", pl.col("To").alias("Node")]),
        ])
        .unique()
        .collect()
    )

    full = (
        nodes.lazy()
        .join(head_a.lazy(), on=["window_start", "Node"], how="left")
        .join(head_b.lazy(), on=["window_start", "Node"], how="left")
        .with_columns(
            # Fill missing: nodes with no outgoing edges => no flow / no currency mix
            pl.col("H_out_1_cnt").fill_null(0.0),
            pl.col("supp_out_1_cnt").fill_null(0).cast(pl.Int64),
            pl.col("pmax_out_1_cnt").fill_null(0.0),

            pl.col("H_out_2_cnt").fill_null(0.0),
            pl.col("supp_out_2_cnt").fill_null(0).cast(pl.Int64),
            pl.col("pmax_out_2_cnt").fill_null(0.0),

            pl.col("n_currencies_out").fill_null(0).cast(pl.Int64),
            pl.col("currency_entropy_out").fill_null(0.0),
            pl.col("top_currency_share_out").fill_null(0.0),
        )
        .collect()
    )

    return full


In [ ]:
train_pos_pred = flow_heads_A_B(ssl)

# Join

In [57]:
final = (
    temporal_features
    # .join(train_pos_pred, on=["window_start", "Node"], how="full", coalesce=True)
    .join(ego_profile, on=["window_start", "Node"], how="full", coalesce=True)
    .join(two_cycles, on=["window_start", "Node"], how="full", coalesce=True)
)

In [58]:
final

window_start,Node,mu_gap_in_sec,var_gap_in_sec,mu_gap_out_sec,var_gap_out_sec,deg_out,fan_out,vol_out,deg_in,fan_in,vol_in,flow_imbalance,n_currencies_in,currency_entropy_in,top_currency_share_in,n_currencies_out,currency_entropy_out,top_currency_share_out,r_2cycle
datetime[μs],u32,f64,f64,f64,f64,i64,i64,f64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64
2022-09-01 00:00:00,1863431,21120.0,1.0993e9,8010.0,5.3427e8,13,4,266870.6,9,3,59082.91,-0.637476,1,-1.0001e-12,1.0,1,-1.0001e-12,1.0,0
2022-09-01 00:00:00,1150128,12150.0,6.216026e8,null,null,0,0,0.0,11,2,5882750.8,1.0,1,-1.0001e-12,1.0,0,0.0,0.0,0
2022-09-07 00:00:00,516338,null,null,97260.0,null,2,1,2444.3,0,0,0.0,-1.0,0,0.0,0.0,1,-9.9964e-13,1.0,0
2022-09-05 00:00:00,1726037,18444.0,1.6380e9,null,null,0,0,0.0,6,1,5388.42,1.0,1,-9.9987e-13,1.0,0,0.0,0.0,0
2022-09-15 00:00:00,998716,35560.0,3.7362e9,300.0,null,2,1,8701.53,4,2,7002.85,-0.108166,1,-9.9987e-13,1.0,1,-9.9987e-13,1.0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2022-09-01 00:00:00,1725707,32760.0,null,17430.0,3.963396e8,5,2,35078.63,2,1,3289.67,-0.828521,1,-9.9987e-13,1.0,1,-1.0001e-12,1.0,0
2022-09-15 00:00:00,381566,null,null,null,null,1,1,96.17,0,0,0.0,-1.0,0,0.0,0.0,1,-9.8965e-13,1.0,0
2022-09-11 00:00:00,987511,44260.0,5.5007e9,6240.0,7.63848e7,3,2,6.4955e6,4,1,241.3,-0.999926,1,-9.9587e-13,1.0,1,-1.0001e-12,1.0,0


# Write

In [59]:
final.write_csv('../data/HI-Medium_SSL.csv')